In [7]:
import pandas as pd

from pandas import ExcelWriter

from time import sleep

import datetime

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from webdriver_manager.chrome import ChromeDriverManager

import os

import zipfile
import requests
import json
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import undetected_chromedriver as uc


#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'AL AFSA' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

#writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = uc.ChromeOptions()
prefs = {
    "plugins.always_open_pdf_externally": True,
    "download.prompt_for_download": False,
    "download.default_directory": tempfolder,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,  # avoid â€œfile may be dangerousâ€ prompt
    "safebrowsing.disable_download_protection": True,  # allow automation downloads
    "profile.default_content_setting_values.automatic_downloads": 1,
}
chromeOptions.add_argument("--disable-search-engine-choice-screen")
chromeOptions.add_experimental_option("prefs", prefs)

#driver = webdriver.Chrome(options=chromeOptions)

import undetected_chromedriver as uc

driver = uc.Chrome(version_main=144, options=chromeOptions)

driver.maximize_window()

# Ensure Chrome can save without prompts (works in recent Chrome)
driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {"behavior": "allow", "downloadPath": tempfolder},
)


Running AL AFSA Web Scraping Tool v.1.1


{}

In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------

regdict= {
    regulatorName+' 1':'https://amf.gov.al/ts_shoqeri_sigurimi.asp',
    regulatorName+' 2':'https://amf.gov.al/ts_shoqeri_risigurimi.asp',
    regulatorName+' 3':'https://amf.gov.al/tt_shoqeri_komisionere.asp',
    regulatorName+' 4':'https://amf.gov.al/tt_rregjistrar.asp',
    regulatorName+' 5':'https://amf.gov.al/tt_treg.asp',
    regulatorName+' 6':'https://amf.gov.al/tsik_fond.asp',
     regulatorName+' 7':'https://amf.gov.al/tsik_depositare.asp',
                   }


Typology={

       regulatorName + ' 1': 'List of "Insurance Companies"',
       regulatorName + ' 2': 'List of "Reinsurance Companies"',
       regulatorName + ' 3': 'List of "Brokerage Companies"',
       regulatorName + ' 4': 'List of "Registrars"',
       regulatorName + ' 5': 'List of "Regulated Markets"',
       regulatorName + ' 6': 'List of "Investment Funds"',
       regulatorName + ' 7': 'List of "Depository for Collective Investment Undertakings"'
        }

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],

		  'Phone - Mother company': [], 'Check': []}



ISO= {"AFGHANISTAN": "AF", "Ã…LAND ISLANDS": "AX", "ALBANIA": "AL", "ALGERIA": "DZ", "AMERICAN SAMOA": "AS", "ANDORRA": "AD", "ANGOLA": "AO", "ANGUILLA": "AI", "ANTARCTICA": "AQ", "ANTIGUA AND BARBUDA": "AG", "ARGENTINA": "AR", "ARMENIA": "AM", "ARUBA": "AW", "AUSTRALIA": "AU", "AUSTRIA": "AT", "AZERBAIJAN": "AZ", "BAHAMAS, THE": "BS", "BAHRAIN": "BH", "BANGLADESH": "BD", "BARBADOS": "BB", "BELARUS": "BY", "BELGIUM": "BE", "BELIZE": "BZ", "BENIN": "BJ", "BERMUDA": "BM", "BHUTAN": "BT", "BOLIVIA": "BO", "BONAIRE, SINT EUSTATIUS AND SABA": "BQ", "BOSNIA AND HERZEGOVINA": "BA", "BOTSWANA": "BW", "BOUVET ISLAND": "BV", "BRAZIL": "BR", "BRITISH INDIAN OCEAN TERRITORY": "IO", "BRUNEI": "BN", "BULGARIA": "BG", "BURKINA FASO": "BF", "BURUNDI": "BI", "CABO VERDE": "CV", "CAMBODIA": "KH", "CAMEROON, UNITED REPUBLIC OF": "CM", "CANADA": "CA", "CAYMAN ISLANDS": "KY", "CENTRAL AFRICAN REPUBLIC": "CF", "CHAD": "TD", "CHILE": "CL", "CHINA, PEOPLES REPUBLIC OF": "CN", "CHRISTMAS ISLAND": "CX", "COCOS (KEELING) ISLANDS": "CC", "COLOMBIA": "CO", "COMOROS": "KM", "CONGO": "CG", "CONGO, DEMOCRATIC REPUBLIC OF THE": "CD", "COOK ISLANDS": "CK", "COSTA RICA": "CR", "CÃ”TE D'IVOIRE": "CI", "CROATIA": "HR", "CUBA": "CU", "CURACAO, BONAIRE, SABA, ST. MARTIN & ST.": "CW", "CYPRUS": "CY", "CZECH REPUBLIC": "CZ", "DENMARK": "DK", "DJIBOUTI": "DJ", "DOMINICA": "DM", "DOMINICAN REPUBLIC": "DO", "ECUADOR": "EC", "EGYPT": "EG", "EL SALVADOR": "SV", "EQUATORIAL GUINEA": "GQ", "ERITREA": "ER", "ESTONIA": "EE", "ESWATINI": "SZ", "ETHIOPIA": "ET", "FALKLAND ISLANDS (MALVINAS)": "FK", "FAROE ISLANDS": "FO", "FIJI": "FJ", "FINLAND": "FI", "FRANCE": "FR", "FRENCH GUIANA": "GF", "FRENCH POLYNESIA": "PF", "FRENCH SOUTHERN TERRITORIES": "TF", "GABON": "GA", "GAMBIA": "GM", "GEORGIA": "GE", "GERMANY": "DE", "GHANA": "GH", "GIBRALTAR": "GI", "GREECE": "GR", "GREENLAND": "GL", "GRENADA": "GD", "GUADELOUPE": "GP", "GUAM": "GU", "GUATEMALA": "GT", "GUERNSEY": "GG", "GUINEA": "GN", "GUINEA-BISSAU": "GW", "GUYANA": "GY", "HAITI": "HT", "HEARD ISLAND AND MCDONALD ISLANDS": "HM", "HOLY SEE": "VA", "HONDURAS": "HN", "HONG KONG": "HK", "HUNGARY": "HU", "ICELAND": "IS", "INDIA": "IN", "INDONESIA": "ID", "IRAN": "IR", "IRAQ": "IQ", "IRELAND": "IE", "ISLE OF MAN": "IM", "ISRAEL": "IL", "ITALY": "IT", "JAMAICA": "JM", "JAPAN": "JP", "JERSEY": "JE", "JORDAN": "JO", "KAZAKHSTAN": "KZ", "KENYA": "KE", "KIRIBATI": "KI", """KOREA (DEMOCRATIC PEOPLE"S REPUBLIC OF)""": "KP", "KOREA, SOUTH": "KR", "KUWAIT": "KW", "KYRGYZSTAN": "KG", "LAO PEOPLE'S DEMOCRATIC REPUBLIC": "LA", "LATVIA": "LV", "LEBANON": "LB", "LESOTHO": "LS", "LIBERIA": "LR", "LIBYA": "LY", "LIECHTENSTEIN": "LI", "LITHUANIA": "LT", "LUXEMBOURG": "LU", "MACAU": "MO", "MADAGASCAR": "MG", "MALAWI": "MW", "MALAYSIA": "MY", "MALDIVES": "MV", "MALI": "ML", "MALTA": "MT", "MARSHALL ISLANDS": "MH", "MARTINIQUE": "MQ", "MAURITANIA": "MR", "MAURITIUS": "MU", "MAYOTTE": "YT", "MEXICO": "MX", "FEDERATED STATES OF MICRONESIA": "FM", "MOLDOVA, REPUBLIC OF": "MD", "MONACO": "MC", "MONGOLIA": "MN", "MONTENEGRO": "ME", "MONTSERRAT": "MS", "MOROCCO": "MA", "MOZAMBIQUE": "MZ", "MYANMAR": "MM", "NAMIBIA": "NA", "NAURU": "NR", "NEPAL": "NP", "NETHERLANDS": "NL", "NEW CALEDONIA": "NC", "NEW ZEALAND": "NZ", "NICARAGUA": "NI", "NIGER": "NE", "NIGERIA": "NG", "NIUE": "NU", "NORFOLK ISLAND": "NF", "NORTH MACEDONIA": "MK", "NORTHERN MARIANA ISLANDS": "MP", "NORWAY": "NO", "OMAN": "OM", "PAKISTAN": "PK", "PALAU": "PW", "PALESTINE, STATE OF": "PS", "PANAMA": "PA", "PAPUA NEW GUINEA": "PG", "PARAGUAY": "PY", "PERU": "PE", "PHILIPPINES": "PH", "PITCAIRN": "PN", "POLAND": "PL", "PORTUGAL": "PT", "PUERTO RICO": "PR", "QATAR": "QA", "RÃ‰UNION": "RE", "ROMANIA": "RO", "RUSSIA": "RU", "RWANDA": "RW", "SAINT BARTHÃ‰LEMY": "BL", "SAINT HELENA, ASCENSION AND TRISTAN DA CUNHA": "SH", "SAINT KITTS AND NEVIS": "KN", "SAINT LUCIA": "LC", "SAINT MARTIN (FRENCH PART)": "MF", "SAINT PIERRE AND MIQUELON": "PM", "SAINT VINCENT AND THE GRENADINES": "VC", "SAMOA": "WS", "SAN MARINO": "SM", "SAO TOME AND PRINCIPE": "ST", "SAUDI ARABIA": "SA", "SENEGAL": "SN", "SERBIA": "RS", "SEYCHELLES": "SC", "SIERRA LEONE": "SL", "SINGAPORE": "SG", "SINT MAARTEN (DUTCH PART)": "SX", "SLOVAKIA": "SK", "SLOVENIA": "SI", "SOLOMON ISLANDS": "SB", "SOMALIA": "SO", "SOUTH AFRICA": "ZA", "SOUTH GEORGIA AND THE SOUTH SANDWICH ISLANDS": "GS", "SOUTH SUDAN": "SS", "SPAIN": "ES", "SRI LANKA": "LK", "SUDAN": "SD", "SURINAME": "SR", "SVALBARD AND JAN MAYEN": "SJ", "SWEDEN": "SE", "SWITZERLAND": "CH", "SYRIAN ARAB REPUBLIC": "SY", "TAIWAN": "TW", "TAJIKISTAN": "TJ", "TANZANIA, UNITED REPUBLIC OF": "TZ", "THAILAND": "TH", "TIMOR-LESTE": "TL", "TOGO": "TG", "TOKELAU": "TK", "TONGA": "TO", "TRINIDAD AND TOBAGO": "TT", "TUNISIA": "TN", "TURKEY": "TR", "TURKMENISTAN": "TM", "TURKS & CAICOS ISLANDS": "TC", "TUVALU": "TV", "UGANDA": "UG", "UKRAINE": "UA", "UNITED ARAB EMIRATES": "AE", "UNITED KINGDOM OF GREAT BRITAIN AND NORTHERN IRELAND": "GB", "UNITED STATES": "US", "UNITED STATES MINOR OUTLYING ISLANDS": "UM", "URUGUAY": "UY", "UZBEKISTAN": "UZ", "VANUATU": "VU", "VENEZUELA": "VE", "VIETNAM": "VN", "BRITISH VIRGIN ISLANDS": "VG", "VIRGIN ISLANDS OF THE U.S.": "VI", "WALLIS AND FUTUNA": "WF", "WESTERN SAHARA": "EH", "YEMEN": "YE", "ZAMBIA": "ZM", "ZIMBABWE": "ZW", "ENGLAND": "GB", "UNITED KINGDOM (OTHER)": "GB", "FRANCE (OTHER)": "FR", "WALES": "GB", "CONGO (KINSHASA)": "CD", "CONGO (BRAZZAVILLE)": "CD", "SCOTLAND": "GB", "ITALY (OTHER)": "IT", "INDONESIA (OTHER)": "ID", "INDIA (OTHER)": "IN", "MOROCCO (OTHER)": "MA", "NEW ZEALAND (OTHER)": "NZ", "SWITZERLAND (OTHER)": "CH", "MALAYSIA (OTHER)": "MY", "NETHERLANDS ANTILLES": "AN", "TRINIDAD & TOBAGO (OTHER)": "TT", "CHANNEL ISLANDS": "GB", "UNITED ARAB EMIRATES (OTHER)": "AE", "DENMARK (OTHER)": "DK", "COMORO ISLANDS": "KM", "MACEDONIA (FORMER YUGOSLAV REPUBLIC OF)": "MK", "SERBIA AND MONTENEGRO(FORMER YUGOSLAVIA)": "CS", "TRINIDAD": "TT", "ETHIOPIA (OTHER)": "ET", "IVORY COAST": "CI", "DUBAI": "AE", "BRITISH WEST INDIES (OTHER)": "VG", "SWAZILAND": "SZ", 'UNITED KINGDOM  (OTHER)': 'GB'}

processdate = now.strftime('%Y-%m-%d')

API_TOKEN = ""
URL = "https://Api.bvdinfo.com/v1/orbis/companies/match"



headers = {
    "ApiToken": API_TOKEN,
    "Content-Type": "application/json"
}
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


def scroll_to_bottom(driver, pause=1.0, max_tries=5):
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(max_tries):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        sleep(pause)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height



In [4]:
#------------------------------------------------ Begin_Main ----------------------------------------
for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    driver.get(regdict[reg])
    sleep(4)

    panels = driver.find_elements(By.CLASS_NAME, "panel-default")
    data_list = []
    for index,panel in enumerate(panels):
        company = panel.find_element(By.TAG_NAME, "strong").text
        # click heading to expand
        sleep(2)
        panel.find_element(By.CLASS_NAME, "panel-heading").click()
        sleep(2)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        sleep(2)
        collapse_id = panel.find_element(By.TAG_NAME, "a").get_attribute("href").split("#")[-1]
            # collapse = driver.find_element(By.ID, f"collapse{i}")
            # print(collapse.text)
        lis_soup = BeautifulSoup(driver.page_source, 'html.parser')
        try:
            lis = lis_soup.find('div',id=collapse_id).find_all('li')
        except:
            sleep(3)
            scroll_to_bottom(driver)
            sleep(3)
            try:
                lis_soup = BeautifulSoup(driver.page_source, 'html.parser')
                lis = lis_soup.find('div',id=f"collapse{index+1}").find_all('li')        
            except:
                # Some taggle maybe missing
                pass 
        data = {"Company": company}
        for li in lis:
            
            row = li.find("div", class_="row")
            key = row.find("div", class_="col-md-3").text.strip().rstrip(":")
            val = row.find("div", class_="col-md-9").text.strip()
            data[key] = val
        data_list.append(data)

    for data in data_list:
        name_ = data.get('Company', '') or ''
        address_ = data.get('Adresa e SelisÃ«', '') or ''
        tel_ = data.get('Tel./Faks', '') or ''
        website_ = data.get('Faqja e Internetit', '') or ''
        email_ = data.get('Posta Elektronike', '') or ''
        register_date = data.get('Licenca') or data.get('Dt. e fillimit tÃ« Aktivitetit') or ''

        if reg == regulatorName + ' 6':
            name_req = name_.replace('FONDI I INVESTIMIT','')
        else:
            name_req = name_
        payload = {
            "MATCH": {
                "Criteria": {
                    # TODO: fill in your matching criteria here
                    # Example:
                    "Name": name_req,
                    "Country": "AL",
                    "EMailOrWebsite": email_ or website_,
                    'Address': address_,
                    'PhoneOrFax': tel_
                },
                "Options": {
                    # TODO: add options if needed
                    "ScoreLimit": 0.85
                    # Greater than 0.85 are good B(0.85-0.94), excellent A (>=0.95)
                }
            },
            "SELECT": [
                # TODO: list fields to return
                # Example:
                # "BvDIDNumber", "Name", "Country", "City", "MatchScore"
            "Match.Hint",  
            "Match.Score",  
            "Match.Name",  
            "Match.Name_Local",  
            "Match.Address",  
            "Match.Postcode",  
            "Match.City",  
            "Match.Country",  
            "Match.Status",  
            "Match.National_Id",  
            "Match.NationalIdLabel",  
            "Match.LegalForm",  
            "Match.BvDId",
            'Match.PhoneOrFax'  ,
            "Match.EmailOrWebsite"
            ]
        }

        response = requests.post(URL, headers=headers, json=payload, timeout=60)
        print("Status:", response.status_code)

        try:
            data = response.json()
            bvd_id=''
            print(json.dumps(data, indent=2))
            if data[0]['Hint'].lower() != 'unlikely':
                bvd_id = data[0]['BvDId'] if data and 'BvDId' in data[0] else ''
            else:
                bvd_id = ''
        except Exception:
            print(response.text)

        # sqldict['bvdid'].append(bvd_id if bvd_id else '')
        sqldict['Name'].append(name_)
        sqldict['ListProcessDate'].append(processdate)
        sqldict['Address_1'].append(address_)
        sqldict['Phone'].append(tel_)
        sqldict['Website'].append(website_)
        sqldict['Email'].append(email_)
        # sqldict['RegulationDate'].append(register_date)
        sqldict['RegCtry'].append(reg.split(' ')[0])
        sqldict['RegCode'].append(reg.split(' ')[1])
        sqldict['ListCode'].append(reg.split(' ')[-1])
        sqldict['RegulationType'].append('Regulated')
        sqldict['ListName'].append(Typology[reg])
    sqldict = bourange_same_length_array(sqldict)
        



[INFO] : Working 1/7 _(AL AFSA 1)_ 
Status: 200
[
  {
    "Hint": "Potential",
    "Score": 0.95,
    "Name": "INSIG SH. A",
    "Name_Local": null,
    "Address": "RRUGA JUL VARIBOBA, NR.21",
    "Postcode": "1031",
    "City": "TIRAN\u00cb",
    "Country": "AL",
    "Status": "Active",
    "National_Id": "L71325019D",
    "NationalIdLabel": "NIPT",
    "LegalForm": null,
    "BvDId": "ALL71325019D",
    "PhoneOrFax": "+355 42400790,+355 42400791",
    "EmailOrWebsite": "info@insig.com.al"
  },
  {
    "Hint": "Potential",
    "Score": 0.88,
    "Name": "INSIG",
    "Name_Local": null,
    "Address": null,
    "Postcode": "8312",
    "City": "LURE",
    "Country": "AL",
    "Status": "Active",
    "National_Id": "J66703717H",
    "NationalIdLabel": "NIPT",
    "LegalForm": "Limited liability company",
    "BvDId": "ALJ66703717H",
    "PhoneOrFax": null,
    "EmailOrWebsite": "insig.com.al"
  },
  {
    "Hint": "Potential",
    "Score": 0.88,
    "Name": "INSIG",
    "Name_Local": null

In [5]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
driver.quit()
sleep(3)

In [6]:

driver = uc.Chrome(version_main=144, options=chromeOptions)

driver.maximize_window()


RuntimeError: you cannot reuse the ChromeOptions object